In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


# USA YouTube Top 1000 - EDA
Simple overview of size, quality, trends, and top channels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

DATA_PATH = '/kaggle/input/datasets/mubashirsidiki/2026-usa-top-1000-youtube-channels-by-subscribers/usa_youtube_top_1000_by_subscribers.csv'
df = pd.read_csv(DATA_PATH)

# Normalize column names to snake_case for stable analysis
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

# Align common alternatives to a simple schema
rename_map = {
    'channel_name': 'title',
    'channel_title': 'title',
}
df = df.rename(columns=rename_map)

print('Shape:', df.shape)
print('Columns:', list(df.columns))
display(df.head(3))


In [ ]:
# Column info + simple quality checks
display(df.info())

quality = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique()
}).sort_values('missing_values', ascending=False)

display(quality)
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# Basic cleaning for analysis
numeric_cols = ['rank', 'views', 'subscribers', 'videos']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

if 'published_at' in df.columns:
    df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
    df['publish_year'] = df['published_at'].dt.year

display(df.describe(include='all').T)


In [ ]:
# Top 10 channels by subscribers
if {'title', 'subscribers'}.issubset(df.columns):
    top10 = df.nlargest(10, 'subscribers')[['title', 'subscribers']]
    display(top10)

    plt.figure(figsize=(10, 5))
    sns.barplot(data=top10, x='subscribers', y='title', hue='title', palette='Blues_r', dodge=False, legend=False)
    plt.title('Top 10 Channels by Subscribers')
    plt.xlabel('Subscribers')
    plt.ylabel('Channel')
    plt.tight_layout()
    plt.show()
else:
    print('Required columns missing for top-10 plot: title and/or subscribers')


In [ ]:
# Distributions for available key metrics
dist_cols = [c for c in ['subscribers', 'views', 'videos'] if c in df.columns]

if dist_cols:
    fig, axes = plt.subplots(1, len(dist_cols), figsize=(5 * len(dist_cols), 4))
    if len(dist_cols) == 1:
        axes = [axes]

    colors = ['teal', 'orange', 'purple']
    for i, col in enumerate(dist_cols):
        sns.histplot(df[col].dropna(), bins=30, ax=axes[i], color=colors[i % len(colors)])
        axes[i].set_title(col.replace('_', ' ').title())

    plt.tight_layout()
    plt.show()
else:
    print('No numeric distribution columns found.')


In [ ]:
# Correlation between numeric columns
corr_cols = [c for c in ['rank', 'subscribers', 'views', 'videos'] if c in df.columns]

if len(corr_cols) >= 2:
    corr = df[corr_cols].corr(numeric_only=True)
    display(corr)

    plt.figure(figsize=(6, 4))
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough numeric columns for correlation heatmap.')


## Summary
- We reviewed data size and quality (missing values, duplicates, types).
- We explored top channels and metric distributions.
- We checked metric relationships and topic category frequency.